In [1]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import yfinance as yf

In [63]:
stock_selected="RELIANCE.NS"
#Reliance chosen in testing mode.The stock or stocks will change according to the Trading strategy. 

In [51]:
stock_data=yf.download(stock_selected,start="2020-01-01",end="2023-08-24")

[*********************100%%**********************]  1 of 1 completed


In [52]:
stock_data

,Open,High,Low,Close,Adj Close,Volume
Date,,,,,,
2020-01-01,1503.745972,1512.760498,1491.363403,1495.424927,1475.384277,6463060
2020-01-02,1497.802368,1526.480469,1497.802368,1520.883545,1500.501587,8173308
2020-01-03,1518.605103,1527.173950,1508.699097,1522.716187,1502.309692,9684434
2020-01-06,1505.727173,1513.552979,1483.933838,1487.400879,1467.467651,11315596
2020-01-07,1504.736572,1520.091064,1499.288208,1510.284058,1490.044312,7699489
...,...,...,...,...,...,...
2023-08-17,2567.100098,2578.100098,2532.850098,2538.000000,2529.066162,6836872
2023-08-18,2531.250000,2577.600098,2508.550049,2556.800049,2547.800049,9319989
2023-08-21,2539.949951,2555.449951,2515.649902,2520.000000,2520.000000,4610873


In [53]:
#Trading strategy
def signalgenerator(df):
    opens=df.Open.iloc[-1];
    close=df.Close.iloc[-1];
    prevopen=df.Open.iloc[-2];
    prevclose=df.Close[-2];
    
    #bearish pattern
    if(opens>close and prevopen<prevclose and opens>=prevclose and close<=prevopen):
        return -1
    #bullish pattern
    elif(opens<close and prevopen>prevclose and opens<=prevclose and close>=prevopen):
        return 1
    else:
        return 0
    
signal=[]
signal.append(0)
for i in range(1,len(stock_data)):
    df=stock_data[i-1:i+1]
    signal.append(signalgenerator(df))
stock_data["signals"]=signal
        

In [54]:
stock_data

,Open,High,Low,Close,Adj Close,Volume,signals
Date,,,,,,,
2020-01-01,1503.745972,1512.760498,1491.363403,1495.424927,1475.384277,6463060,0
2020-01-02,1497.802368,1526.480469,1497.802368,1520.883545,1500.501587,8173308,0
2020-01-03,1518.605103,1527.173950,1508.699097,1522.716187,1502.309692,9684434,0
2020-01-06,1505.727173,1513.552979,1483.933838,1487.400879,1467.467651,11315596,0
2020-01-07,1504.736572,1520.091064,1499.288208,1510.284058,1490.044312,7699489,0
...,...,...,...,...,...,...,...
2023-08-17,2567.100098,2578.100098,2532.850098,2538.000000,2529.066162,6836872,0
2023-08-18,2531.250000,2577.600098,2508.550049,2556.800049,2547.800049,9319989,0
2023-08-21,2539.949951,2555.449951,2515.649902,2520.000000,2520.000000,4610873,0


In [55]:
stock_data.signals.value_counts()

 0    834
-1     50
 1     22
Name: signals, dtype: int64

In [56]:
#backtest of strategy
def backtest(amount,stock_data):
    current_stocks=0
    signal=0
    for i in range(1,len(stock_data)):
        
        
        if(signal==1):
            
            price=stock_data.Open.iloc[i]
            qt=amount/price
            current_stocks+=qt
            amount-=(qt*price)
        elif(signal==-1):
            price=stock_data.Open.iloc[i]
            amount+=current_stocks*price
            current_stocks=0
        signal=stock_data.signals.iloc[i]
        
    if(current_stocks>0):
        price=stock_data.Open.iloc[-1]
        amount+=current_stocks*price
    return amount
    
            

In [57]:
finalamount=backtest(1000000,stock_data)

In [58]:
finalamount

1811681.4320006636

In [2]:
#Connect to market
import alpaca_trade_api as api
import time 
import random

In [3]:
API_KEY = "PKNVSU3OUT5WCAQD7DL6"
API_SECRET = "IZHh081kfu69wrP9BMb2Oeu75MyVF51572K4crwX"
BASE_URL = "https://paper-api.alpaca.markets"

alpaca = api.REST(API_KEY, API_SECRET, BASE_URL)


In [ ]:
#Placing order
pos_held = False
symb='RELFF'

while True:
    
    signal=stock_data.signals.iloc[-1]
    if(signal==1 and not pos_held):
        api.submit_order(
            symbol=symb,
            qty=1,
            side='buy',
            type='market',
            time_in_force='gtc'
        )
    elif(signal==-1 and pos_held):
        api.submit_order(
            symbol=symb,
            qty=1,
            side='sell',
            type='market',
            time_in_force='gtc'
        )
    time.sleep(60*24)
        
    
    